### Plot a PCA for Spheroid Aggregated data

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Grit scores
# from cytominer_eval import evaluate

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns; sns.set_style("white")

# Set current working directory


In [ ]:
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

In [ ]:
# Save the data
ImagesOut = str(figdir('SupplFig4')) + '/'

# One output per acquisition. NOTE: the published Suppl 4h/4i "air" panel is the two
# non-WI acquisitions COMBINED (see PORT_TRIAGE); this notebook emits them separately.
PLATE_TAG = {
    'CellPainting_20241220clearedspheroidsBOMI_20241220_151510': 'bomi',
    'CellPainting_20250127Cellpaintcleared3D_20250127_171120':   'cleared3d',
    'CellPainting_CellPaint3DBomi_WI_for_Jordi_20250203_155142': 'wi',
}


if not os.path.exists(ImagesOut): 
        os.makedirs(ImagesOut)

In [ ]:
color_dict_controls = {
    'water': '#66c2a5',
    'DMSO': '#b3b3b3',
    'sorbitol': '#e5c494',
    'fluphenazine': '#a6d854',
    'fenbendazole': '#ffd92f',
    'etoposide': '#fc8d62',
    'berberine chloride': '#e78ac3',
    'nocodazole': '#66c2a5',
}

In [ ]:
cell_line = 'HT29'

In [ ]:
# # Load the data
dir = str(profiles("exp4_objective", "")) + "/"
data =  pd.read_parquet(('{}selected_data_{}.parquet').format(dir, cell_line))

In [ ]:
# Some function definitions

def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

### PCA

In [ ]:
plates = data.Metadata_Barcode.unique()

plate = plates[0]  

dataset = data.query('Metadata_Barcode == @plate')

# Get the features
ListOfFeatures, list_of_metadata = list_features(dataset)

In [ ]:
dataset.Metadata_cmpd_conc.astype(str).unique()

concs_to_drop = ['0.0316473', '0.010002279', '0.003167009', '0.001002506',
       '0.000316388']

# concs_to_drop = ['0.0316473', '0.010002279']

In [ ]:
# Run a PCA
from sklearn.decomposition import PCA

## PCA - Color by batch

# Filter by stronger compounds
posconDf = dataset.copy()
# Remove inactive compounds water and sorbitol
posconDf = posconDf.query('Metadata_cmpdname != "water"')
# posconDf = posconDf.query('Metadata_cmpdname != "etoposide"')
# posconDf = posconDf.query('Metadata_cmpdname != "nocodazole"')
# posconDf = posconDf.query('Metadata_cmpdname != "fenbendazole"')
posconDf = posconDf.query('Metadata_cmpdname != "sorbitol"')

# Remove the lower concentrations
for conc in concs_to_drop:
    posconDf = posconDf.query('Metadata_cmpd_conc != @conc')

# Replace nans with 0
posconDf.loc[:, ListOfFeatures[:]] = posconDf.loc[:, ListOfFeatures[:]].fillna(0)
training_data = posconDf.loc[:, ListOfFeatures[:]].values

pca = PCA(n_components=4)
pca.fit(training_data)
pca_embedding = pca.transform(training_data)


## Plotting
fig = plt.figure(figsize=(3, 3))
ax = sns.scatterplot(
    x=pca_embedding[:, 0],
    y=pca_embedding[:, 1],
    hue=posconDf.Metadata_cmpdname,
    alpha=(0.7),
    s=30,
    linewidth=0,
    legend=True,
    palette=color_dict_controls,
    )

# # ax.title.set_text(cell_line)
ax.legend(bbox_to_anchor=(1.1, 1), loc=2, borderaxespad=0.0, fontsize=10)
plt.title(plate)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

# Save the plot
fig.savefig("{}/{}_{}_{}.{}".format(dir, "PCA", 'unlabeled', plate, 'pdf'),
            dpi=300,
            bbox_inches='tight')

In [ ]:
# Run a UMAP
from umap import UMAP
import umap.plot

reducer = umap.UMAP(n_components=2, n_neighbors=6, random_state=42, metric='euclidean', min_dist=0.01, spread=2)
embedding = reducer.fit_transform(training_data)

## UMAP - Pathway

fig = plt.figure(figsize=(3,3))
ax = sns.scatterplot(
    x=embedding[:, 0],
    y=embedding[:, 1],
    hue=posconDf.Metadata_cmpdname,
    alpha=(0.7),
    marker="o",
    #size=df_toplot["Metadata_cmpd_conc"],
    palette=color_dict_controls,
    s=30,
    linewidth=0,
    legend=True
    )

plt.title('UMAP: ' + plate)
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
ax.legend(bbox_to_anchor=(1.1, 1), loc=2, borderaxespad=0.0, fontsize=10)
plt.show()

# Save the plot
save_panel(fig, f'SupplFig4h_unlab{PLATE_TAG[plate]}',
           data=pd.DataFrame({'umap1': embedding[:, 0], 'umap2': embedding[:, 1],
                              'compound': posconDf.Metadata_cmpdname.values}),
           caption=f'Unsupervised UMAP, {plate}',
           notebook='analysis/3_SupplFigure4/3_PCA_objective.ipynb')

In [ ]:
## Run a labeled UMAP
# Create a mapping for one-hot encoding

def oneHot(row, mapping):
    return mapping.get(row, -1)

# Generate one-hot encoding for 'Metadata_cmpdname'
onehot_list = posconDf['Metadata_cmpdname'].unique()
onehot_mapping = {name: i for i, name in enumerate(onehot_list)}

# Apply one-hot encoding and add it as a new column
posconDf['Metadata_cmpd_onehot'] = posconDf['Metadata_cmpdname'].apply(lambda name: oneHot(name, onehot_mapping))

reducer = umap.UMAP(n_components=2, n_neighbors=10, random_state=42, metric='euclidean', min_dist=0.1, spread=5)
embedding = reducer.fit_transform(training_data, y=posconDf['Metadata_cmpd_onehot'].to_list())

## UMAP - Pathway

fig = plt.figure(figsize=(3,3))
ax = sns.scatterplot(
    x=embedding[:, 0],
    y=embedding[:, 1],
    hue=posconDf.Metadata_cmpdname,
    alpha=(0.7),
    marker="o",
    s=30,
    linewidth=0,
    palette=color_dict_controls,
    legend=True
    )

plt.title('UMAP - Labeled :' + plate)
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
ax.legend(bbox_to_anchor=(1.1, 1), loc=2, borderaxespad=0.0, fontsize=10)
plt.show()

# Save the plot
save_panel(fig, f'SupplFig4h_lab{PLATE_TAG[plate]}',
           data=pd.DataFrame({'umap1': embedding[:, 0], 'umap2': embedding[:, 1],
                              'compound': posconDf.Metadata_cmpdname.values}),
           caption=f'Labelled UMAP, {plate}',
           notebook='analysis/3_SupplFigure4/3_PCA_objective.ipynb')